### Часть 1: Предобработка и анализ текста
В этом разделе мы выполняем:
1. Токенизацию
2. Частеречную разметку (POS)
3. Лемматизацию
4. Распознавание именованных сущностей (NER)
5. Разбор предложения (зависимости)

In [34]:
import spacy
from spacy import displacy

# Загрузка модели
nlp = spacy.load('en_core_web_sm')

# Пример текста
text = "Apple is looking at buying U.K. startup for $1 billion. Steve Jobs founded the company in California."
doc = nlp(text)

# 1, 2, 3. Токенизация, частиречная разметка и лемматизация
print(f"{'Токен':<15} | {'Часть речи':<12} | {'Лемма':<15}")
print("-" * 45)
for token in doc[:10]:
    print(f"{token.text:<15} | {token.pos_:<12} | {token.lemma_:<15}")

Токен           | Часть речи   | Лемма          
---------------------------------------------
Apple           | PROPN        | Apple          
is              | AUX          | be             
looking         | VERB         | look           
at              | ADP          | at             
buying          | VERB         | buy            
U.K.            | PROPN        | U.K.           
startup         | VERB         | startup        
for             | ADP          | for            
$               | SYM          | $              
1               | NUM          | 1              


In [35]:
# 4. Распознавание именованных сущностей (NER)
print("Именованные сущности:")
for ent in doc.ents:
    print(f"{ent.text:<15} | {ent.label_}")

# Визуализация сущностей
displacy.render(doc, style='ent', jupyter=True)

Именованные сущности:
Apple           | ORG
U.K.            | GPE
$1 billion      | MONEY
Steve Jobs      | PERSON
California      | GPE


In [36]:
# 5. Разбор предложения (синтаксические связи)
print("Визуализация синтаксического разбора:")
displacy.render(next(doc.sents), style='dep', jupyter=True, options={'distance': 100})

Визуализация синтаксического разбора:


### Часть 2: Классификация текстов

В этой части мы сравним два метода векторизации текста для задачи классификации на наборе данных `20newsgroups`.

In [37]:
!pip install gensim nltk scikit-learn

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize

# Загружаем необходимые ресурсы NLTK
nltk.download('punkt')
nltk.download('punkt_tab')

# Загружаем подмножество данных
categories = ['sci.med', 'sci.space', 'talk.politics.guns', 'rec.autos']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

X_train, X_test, y_train, y_test = train_test_split(newsgroups.data, newsgroups.target, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Размер обучающей выборки: 3101
Размер тестовой выборки: 776


#### Способ 1: Использование TfidfVectorizer

In [38]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = model_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print(f"Точность TF-IDF: {acc_tfidf:.4f}")

Точность TF-IDF: 0.8737


#### Способ 2: Использование Word2Vec

In [39]:
# Токенизация для Word2Vec
tokenized_train = [word_tokenize(text.lower()) for text in X_train]
tokenized_test = [word_tokenize(text.lower()) for text in X_test]

# Обучение Word2Vec
w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=1, workers=4)

# Функция для получения среднего вектора текста
def get_mean_vector(model, words):
    words = [word for word in words if word in model.wv]
    if len(words) >= 1:
        return np.mean(model.wv[words], axis=0)
    return np.zeros((model.vector_size,))

X_train_w2v = np.array([get_mean_vector(w2v_model, t) for t in tokenized_train])
X_test_w2v = np.array([get_mean_vector(w2v_model, t) for t in tokenized_test])

model_w2v = LogisticRegression(max_iter=1000)
model_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = model_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test, y_pred_w2v)

print(f"Точность Word2Vec: {acc_w2v:.4f}")

Точность Word2Vec: 0.5129


#### Сравнение результатов

In [40]:
print(f"Точность TF-IDF: {acc_tfidf:.4f}")
print(f"Точность Word2Vec: {acc_w2v:.4f}")

if acc_tfidf > acc_w2v:
    print("Вывод: Метод TF-IDF показал более высокую точность.")
else:
    print("Вывод: Метод Word2Vec показал более высокую точность.")

Точность TF-IDF: 0.8737
Точность Word2Vec: 0.5129
Вывод: Метод TF-IDF показал более высокую точность.


### Часть 2: Классификация текстов

В этой части мы сравним два метода векторизации текста для задачи классификации на наборе данных `20newsgroups`.

In [41]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

# Загружаем подмножество данных
categories = ['sci.med', 'sci.space', 'talk.politics.guns', 'rec.autos']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

X_train, X_test, y_train, y_test = train_test_split(newsgroups.data, newsgroups.target, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Размер обучающей выборки: 3101
Размер тестовой выборки: 776


#### Способ 1: Использование TfidfVectorizer

In [42]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = model_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print(f"Точность TF-IDF: {acc_tfidf:.4f}")

Точность TF-IDF: 0.8737


#### Способ 2: Использование Word2Vec

In [43]:
# Токенизация для Word2Vec
tokenized_train = [word_tokenize(text.lower()) for text in X_train]
tokenized_test = [word_tokenize(text.lower()) for text in X_test]

# Обучение Word2Vec
w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=1, workers=4)

# Функция для получения среднего вектора текста
def get_mean_vector(model, words):
    words = [word for word in words if word in model.wv]
    if len(words) >= 1:
        return np.mean(model.wv[words], axis=0)
    return np.zeros((model.vector_size,))

X_train_w2v = np.array([get_mean_vector(w2v_model, t) for t in tokenized_train])
X_test_w2v = np.array([get_mean_vector(w2v_model, t) for t in tokenized_test])

model_w2v = LogisticRegression(max_iter=1000)
model_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = model_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test, y_pred_w2v)

print(f"Точность Word2Vec: {acc_w2v:.4f}")

Точность Word2Vec: 0.5052


#### Сравнение результатов

In [44]:
print(f"Точность TF-IDF: {acc_tfidf:.4f}")
print(f"Точность Word2Vec: {acc_w2v:.4f}")

if acc_tfidf > acc_w2v:
    print("Вывод: Метод TF-IDF показал более высокую точность.")
else:
    print("Вывод: Метод Word2Vec показал более высокую точность.")

Точность TF-IDF: 0.8737
Точность Word2Vec: 0.5052
Вывод: Метод TF-IDF показал более высокую точность.


### Часть 2: Классификация текстов

В этой части мы сравним два метода векторизации текста для задачи классификации на наборе данных `20newsgroups`.

In [45]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

# Загружаем небольшой поднабор данных для ускорения
categories = ['sci.med', 'sci.space', 'talk.politics.guns', 'rec.autos']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

X_train, X_test, y_train, y_test = train_test_split(newsgroups.data, newsgroups.target, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Размер обучающей выборки: 3101
Размер тестовой выборки: 776


#### Способ 1: TfidfVectorizer + Логистическая регрессия

In [46]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = model_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("Результаты TF-IDF:")
print(f"Accuracy: {acc_tfidf:.4f}")

Результаты TF-IDF:
Accuracy: 0.8737


#### Способ 2: Word2Vec + Логистическая регрессия
Для классификации мы будем использовать среднее значение векторов всех слов в предложении.

In [47]:
# Токенизация для Word2Vec
tokenized_train = [word_tokenize(text.lower()) for text in X_train]
tokenized_test = [word_tokenize(text.lower()) for text in X_test]

# Обучение модели Word2Vec
w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=1, workers=4)

# Функция для усреднения векторов
def get_mean_vector(model, words):
    words = [word for word in words if word in model.wv]
    if len(words) >= 1:
        return np.mean(model.wv[words], axis=0)
    return np.zeros((model.vector_size,))

X_train_w2v = np.array([get_mean_vector(w2v_model, tokens) for tokens in tokenized_train])
X_test_w2v = np.array([get_mean_vector(w2v_model, tokens) for tokens in tokenized_test])

model_w2v = LogisticRegression(max_iter=1000)
model_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = model_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test, y_pred_w2v)

print("Результаты Word2Vec:")
print(f"Accuracy: {acc_w2v:.4f}")

Результаты Word2Vec:
Accuracy: 0.5064


#### Сравнение результатов

In [48]:
print(f"Точность TF-IDF: {acc_tfidf:.4f}")
print(f"Точность Word2Vec: {acc_w2v:.4f}")

if acc_tfidf > acc_w2v:
    print("Вывод: В данной задаче TF-IDF показал лучший результат.")
else:
    print("Вывод: В данной задаче Word2Vec показал лучший результат.")

Точность TF-IDF: 0.8737
Точность Word2Vec: 0.5064
Вывод: В данной задаче TF-IDF показал лучший результат.


### Интерпретация результатов

На основе проведенных экспериментов можно сделать следующие выводы:

1. **Превосходство TF-IDF:** В данной задаче метод TF-IDF показал значительно более высокую точность (около 87%) по сравнению с Word2Vec (около 51%). Это объясняется тем, что для классификации по темам (как в наборе 20 Newsgroups) наличие специфических ключевых слов является определяющим фактором, который TF-IDF эффективно учитывает.

2. **Особенности Word2Vec:**
    * Низкая точность Word2Vec связана с тем, что модель обучалась «с нуля» на очень маленьком подмножестве данных. Для качественного построения семантического пространства векторов требуется огромный корпус текстов.
    * Метод усреднения векторов слов («Bag of Vectors») теряет информацию о порядке слов и синтаксической структуре, что также ограничивает точность.

3. **Рекомендации:** Для улучшения классификации на базе нейросетевых моделей рекомендуется использовать предобученные эмбеддинги (Glove, FastText) или современные трансформерные модели (BERT), которые уже обладают знаниями о языке.